# AAOS Agent — Colab runner (vLLM + Qwen)

Runs the agent on Colab with **vLLM** serving **Qwen2.5-Coder-32B** (AWQ), project cloned
from GitHub, AOSP repos shallow-cloned by release tag, index persisted on **Google Drive**.

Order (top→bottom):
0. GPU + Drive
1. Reusable vLLM backend function (`ensure_llm()`)
2. Clone project from GitHub + deps
3. Config (point agent at vLLM, stores→Drive)
4. Choose SCOPE → shallow-clone AOSP by tag → index → Drive (skips if built)
5. Run agent

**Re-run / recovery**: call `ensure_llm()` again anytime the server dies — it restarts
vLLM only if needed. Index (Drive) is reused across sessions.

## 0 — GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/aaos'
STORES=f'{DRIVE}/stores'          # only the INDEX persists to Drive (not the model)
os.makedirs(STORES,exist_ok=True)
print('STORES =',STORES)

## 1 — Reusable vLLM backend function

`ensure_llm()` is **idempotent**: checks whether vLLM is already serving on :8000 and only
(re)starts it if needed. Call it here, and call it again any time the model fails or you
re-run — it brings vLLM back without redoing setup.

In [ ]:
import os, subprocess, time, requests

MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct-AWQ'   # AWQ fits A100 80GB with room for KV
VLLM_URL = 'http://127.0.0.1:8000'

def _alive(path='/health'):
    try: return requests.get(VLLM_URL+path, timeout=2).ok
    except Exception: return False

def ensure_llm(max_wait=900):
    """Idempotent: start vLLM (Qwen) if not already serving; return its base_url.
    Safe to call again after a failure or on re-run."""
    if not _alive():
        # install once (no-op if present)
        subprocess.run('pip -q install vllm', shell=True)
        cmd=(f'python -m vllm.entrypoints.openai.api_server --model {MODEL} '
             '--quantization awq --dtype half --gpu-memory-utilization 0.90 '
             '--max-model-len 16384 --port 8000')
        subprocess.Popen(cmd+' > /content/vllm.log 2>&1', shell=True, env={**os.environ})
        for i in range(max_wait//10):
            if _alive(): break
            time.sleep(10); print(f'  vLLM starting… {(i+1)*10}s')
        else:
            print('vLLM NOT ready — log tail:')
            print(''.join(open('/content/vllm.log').readlines()[-25:]))
    print('vLLM up' if _alive() else 'vLLM DOWN')
    return VLLM_URL+'/v1'

API_BASE = ensure_llm()
print('LLM ready at', API_BASE)

## 2 — Clone project from GitHub + deps

In [ ]:
import os
if not os.path.isdir('/content/android-auto-ai-agent'):
    !git clone https://github.com/appdev1307/android-auto-ai-agent.git /content/android-auto-ai-agent
else:
    !cd /content/android-auto-ai-agent && git pull
%cd /content/android-auto-ai-agent
!apt-get -qq install -y ripgrep >/dev/null && echo ripgrep ok
!pip -q install -r requirements.txt
print('project ready')

## 3 — Config: point agent at vLLM, stores→Drive

In [ ]:
import yaml, pathlib, os
cfg_path=pathlib.Path('data/config.yaml'); cfg=yaml.safe_load(cfg_path.read_text())
cfg['model']['name']=MODEL
cfg['model']['api_base']=API_BASE          # from ensure_llm()
cfg['rag']['stores_root']=STORES
cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
os.environ['OPENAI_API_KEY']='dummy'       # vLLM ignores it
print('config →',MODEL,'@',API_BASE)

## 4 — Choose SCOPE → shallow-clone AOSP by tag → index → Drive

`SCOPE` drives BOTH what gets cloned and what gets indexed (no code edits):
`automotive` (AAOS stack) · `framework` (+ frameworks/base, fuller full-stack) · `full`.
Repos are shallow-cloned by release tag from android.googlesource.com (like the thesis).

In [ ]:
SCOPE = 'framework'          # 'automotive' | 'framework' | 'full'
TAG   = 'android-15.0.0_r1'  # AOSP release tag to pin

import os, subprocess, pathlib
AOSP='/content/aosp'
# dest path (in the tree layout the indexer expects)  ->  googlesource repo
REPOS = {
  'hardware/interfaces':     'platform/hardware/interfaces',
  'packages/services/Car':   'platform/packages/services/Car',
  'packages/apps/Car':       'platform/packages/apps/Car',
  'frameworks/base':         'platform/frameworks/base',
}
SCOPE_REPOS = {
  'automotive': ['hardware/interfaces','packages/services/Car','packages/apps/Car'],
  'framework' : ['hardware/interfaces','packages/services/Car','packages/apps/Car','frameworks/base'],
  'full'      : [],   # provide a full checkout yourself
}
def clone(sub, dest):
    if os.path.isdir(dest): print('  have',dest); return
    os.makedirs(pathlib.Path(dest).parent, exist_ok=True)
    subprocess.run(['git','clone','--depth=1','-b',TAG,
                    f'https://android.googlesource.com/{sub}', dest], check=True)
for rel in SCOPE_REPOS.get(SCOPE, []):
    clone(REPOS[rel], f'{AOSP}/{rel}')
os.environ['AOSP_ROOT']=AOSP
print('SCOPE =',SCOPE,'| TAG =',TAG,'| tree at',AOSP)
!du -sh {AOSP} 2>/dev/null

In [ ]:
import os
if os.path.exists(f'{STORES}/_base/aosp15/manifest.json'):
    print('index already on Drive → skip (delete manifest to rebuild / change scope)')
else:
    # embedder on CPU (vLLM holds the GPU); index into the chosen scope
    !CUDA_VISIBLE_DEVICES="" python -m retrieval.indexer --aosp-root {AOSP} --base --aosp-version aosp15 --scope {SCOPE}
print('index at', f'{STORES}/_base/aosp15')

## 5 — Run agent

In [ ]:
BUG="Android 15: VSS Vehicle.Speed not updating in HMI after ignition ON"
!CUDA_VISIBLE_DEVICES="" python -m agent.main --bug "{BUG}" --aosp-root {AOSP} --aosp-version aosp15

---
### Re-run / recovery
- **vLLM died / idle-killed** → re-run the `ensure_llm()` cell (or call `ensure_llm()`
  anywhere). It re-checks :8000 and restarts vLLM only if needed. No full redo.
- **New session** → cells 0,1,2,3 then 5. Index (Drive) is reused; cell 4 re-clones the
  small AOSP subset (or skip it and run index-only on dense+BM25).
- **Change coverage** → set `SCOPE` in cell 4, delete `stores/_base/aosp15/manifest.json`, re-run cell 4.
- **OOM** → lower `--max-model-len` to 8192 or `--gpu-memory-utilization` to 0.85 in `ensure_llm()`.